# NBA Comp Viz: SportVU 2015-16 analytics dataset on Drive (bootstrap)

**One cell.** All run logic lives in the repo (`sportvu/colab_contract.py` on branch `track-fix`), pulled fresh by
`git reset --hard origin/<branch>` every run, so a stale copy of this notebook cannot run stale code.
The schema is `docs/DATA_CONTRACT.md`.

1. No GPU needed (Runtime > Change runtime type > CPU is fine). Disk and network are what matter.
2. Run the cell. When Google asks which account to connect to Drive, pick **lucienmmcnulty@gmail.com**.
   The run stops at once if a different Drive is mounted (it checks `My Drive/nba_harvest/sportvu/ACCOUNT.txt`).
3. About 636 games at roughly 10 to 15 s each: 2 to 3 hours. Output lands in
   `My Drive/nba_harvest/sportvu_contract/` (one parquet per game, `heldout/` for the held-out games,
   `manifest.json` after every game, `README.md`). If Colab disconnects, run the cell again: games already
   on Drive are skipped.
4. The last line prints `RUN OK` or `RUN FAILED` (a failed run lists the games it could not convert;
   `manifest.json` has the reason per game).


In [ ]:
#@title Build the SportVU contract dataset on Drive (bootstrap: all logic is in the repo)
BRANCH = "track-fix"        # the branch that carries sportvu/colab_contract.py
EXTRA_ARGS = ""             # full build. "--limit N" for a smoke run; "--retry-pbp" fills in play-by-play that failed
import os
from google.colab import drive
drive.mount("/content/drive")
if not os.path.exists("/content/repo"):
    !git clone --depth 1 --branch $BRANCH https://github.com/lmcnulty7/NBA-Comp-Viz-V3.git /content/repo
%cd /content/repo
!git fetch --depth 1 origin $BRANCH && git reset --hard origin/$BRANCH && git log --oneline -1
!rm -f /content/run_status.txt
!python -m sportvu.colab_contract $EXTRA_ARGS
# Block until Drive has uploaded everything (a run that skipped this lost files in the async upload).
drive.flush_and_unmount()
status = open("/content/run_status.txt").read() if os.path.exists("/content/run_status.txt") else "RUN FAILED: the runner stopped before writing a status (read the output above)"
print(status + " | Drive flushed and unmounted.")
